# Text8 LSTM Training and Tokenizer Comparison on Google Colab

This notebook trains a baseline **LSTM language model** on **Text8** and compares **word-level**, **character-level**, and **BPE** tokenization using the same project codebase.

Recommended runtime:
- `Runtime` -> `Change runtime type` -> `GPU`
- Colab T4 / L4 is a good default choice


## What is BPE?

**BPE (Byte Pair Encoding)** is a subword tokenization method.

Instead of representing a sentence only as full words or only as characters, BPE learns frequent subword chunks such as `play`, `ing`, or `tion`.

Why it matters for this assignment:
- It reduces OOV problems compared with pure word-level tokenization.
- It keeps sequences shorter than character-level tokenization.
- It often provides a better trade-off between vocabulary size, sequence length, and perplexity.

In this notebook, the current training pipeline supports:
- `word`
- `char`
- `bpe`


In [ ]:
from pathlib import Path
import os
import sys

IS_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/HatakekkSheeshh/text-preprocess-tokenization.git"
BRANCH = "main"
REPO_DIR = Path("/content/text-preprocess-tokenization")

if IS_COLAB and not REPO_DIR.exists():
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}

if IS_COLAB:
    os.chdir(REPO_DIR)

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
import json
import math
import matplotlib.pyplot as plt
import torch

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.datasets.load_data import load
from src.training.train_lstm import LSTMTrainingConfig, train_lstm_language_model

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Download/load Text8 into data/raw/text8 if needed.
load("text8")
print("Text8 is ready.")


## Configure a single LSTM run

Use this section when you want to train one tokenizer in depth.

Recommended starting points:
- `medium` for a first meaningful run
- `full` for a stronger experiment on Colab GPU


In [ ]:
SINGLE_RUN_PRESETS = {
    "smoke": {
        "sequence_length": 32,
        "batch_size": 8,
        "embedding_dim": 32,
        "hidden_dim": 64,
        "num_layers": 1,
        "dropout": 0.1,
        "epochs": 1,
        "learning_rate": 1e-3,
        "max_vocab_size": 5000,
        "max_train_tokens": 4096,
        "max_validation_tokens": 1024,
        "max_test_tokens": 1024,
        "log_interval": 20,
    },
    "medium": {
        "sequence_length": 128,
        "batch_size": 32,
        "embedding_dim": 128,
        "hidden_dim": 256,
        "num_layers": 2,
        "dropout": 0.2,
        "epochs": 3,
        "learning_rate": 1e-3,
        "max_vocab_size": 20000,
        "max_train_tokens": 2000000,
        "max_validation_tokens": 250000,
        "max_test_tokens": 250000,
        "log_interval": 100,
    },
    "full": {
        "sequence_length": 128,
        "batch_size": 64,
        "embedding_dim": 256,
        "hidden_dim": 512,
        "num_layers": 2,
        "dropout": 0.2,
        "epochs": 5,
        "learning_rate": 1e-3,
        "max_vocab_size": 50000,
        "max_train_tokens": None,
        "max_validation_tokens": None,
        "max_test_tokens": None,
        "log_interval": 200,
    },
}

COMPARE_PRESETS = {
    "smoke": {
        "sequence_length": 32,
        "batch_size": 8,
        "embedding_dim": 32,
        "hidden_dim": 64,
        "num_layers": 1,
        "dropout": 0.1,
        "epochs": 1,
        "learning_rate": 1e-3,
        "max_vocab_size": 5000,
        "max_train_tokens": 4096,
        "max_validation_tokens": 1024,
        "max_test_tokens": 1024,
        "log_interval": 20,
    },
    "medium": {
        "sequence_length": 128,
        "batch_size": 32,
        "embedding_dim": 128,
        "hidden_dim": 256,
        "num_layers": 2,
        "dropout": 0.2,
        "epochs": 2,
        "learning_rate": 1e-3,
        "max_vocab_size": 20000,
        "max_train_tokens": 300000,
        "max_validation_tokens": 50000,
        "max_test_tokens": 50000,
        "log_interval": 100,
    },
}

def make_config(tokenizer_name, preset_name, *, compare_mode=False, run_suffix=None):
    preset_table = COMPARE_PRESETS if compare_mode else SINGLE_RUN_PRESETS
    settings = preset_table[preset_name].copy()

    if tokenizer_name == "char":
        settings["max_vocab_size"] = None

    run_name = run_suffix or f"colab_text8_lstm_{tokenizer_name}_{preset_name}"

    return LSTMTrainingConfig(
        dataset_name="text8",
        tokenizer_name=tokenizer_name,
        device="cuda" if torch.cuda.is_available() else "cpu",
        num_workers=2,
        run_name=run_name,
        **settings,
    )

SINGLE_RUN_PROFILE = "full"   # smoke, medium, full
TOKENIZER_NAME = "word"       # word, char, bpe

single_run_config = make_config(TOKENIZER_NAME, SINGLE_RUN_PROFILE, compare_mode=False)
single_run_config


## Train one tokenizer deeply


In [ ]:
single_summary = train_lstm_language_model(single_run_config)
single_summary["test"]


In [ ]:
single_metrics_path = PROJECT_ROOT / "outputs" / "metrics" / "lstm" / f"{single_summary['run_name']}.json"
single_metrics = json.loads(single_metrics_path.read_text(encoding="utf-8"))

print("Metrics file:", single_metrics_path)
print("Checkpoint dir:", single_metrics["checkpoint_dir"])
print("Best validation loss:", single_metrics["best_validation_loss"])
print("Test metrics:")
print(json.dumps(single_metrics["test"], indent=2))


In [ ]:
history = single_metrics["history"]
epochs = [item["epoch"] for item in history]
train_losses = [item["train"]["loss"] for item in history]
val_losses = [item["validation"]["loss"] for item in history]
train_ppl = [item["train"]["perplexity"] for item in history]
val_ppl = [item["validation"]["perplexity"] for item in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_losses, marker="o", label="Train loss")
axes[0].plot(epochs, val_losses, marker="o", label="Validation loss")
axes[0].set_title("Loss by Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()

axes[1].plot(epochs, train_ppl, marker="o", label="Train perplexity")
axes[1].plot(epochs, val_ppl, marker="o", label="Validation perplexity")
axes[1].set_title("Perplexity by Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Perplexity")
axes[1].legend()

plt.tight_layout()
plt.show()


## Compare word vs char vs BPE

This section runs three experiments with the same LSTM architecture and compares them.

Recommended preset:
- `medium` for a practical Colab comparison
- `smoke` for a fast functionality check

The character-level run can still be slower because it produces much longer token sequences.

Important note: raw perplexity across `word`, `char`, and `bpe` is not perfectly apples-to-apples because the token spaces are different. In the report, interpret perplexity together with vocabulary size, sequence length, and training time.


In [ ]:
COMPARISON_PROFILE = "medium"  # smoke or medium
COMPARISON_TOKENIZERS = ["word", "char", "bpe"]

comparison_runs = []

for tokenizer_name in COMPARISON_TOKENIZERS:
    print(f"\n===== Running tokenizer: {tokenizer_name} =====")
    config = make_config(
        tokenizer_name,
        COMPARISON_PROFILE,
        compare_mode=True,
        run_suffix=f"colab_compare_{tokenizer_name}_{COMPARISON_PROFILE}",
    )
    summary = train_lstm_language_model(config)
    comparison_runs.append(summary)

comparison_runs


In [ ]:
comparison_rows = []

for run in comparison_runs:
    last_epoch = run["history"][-1]
    comparison_rows.append(
        {
            "tokenizer": run["tokenizer"]["type"],
            "vocab_size": run["tokenizer"]["vocab_size"],
            "train_loss": last_epoch["train"]["loss"],
            "val_loss": last_epoch["validation"]["loss"],
            "val_ppl": last_epoch["validation"]["perplexity"],
            "test_ppl": run["test"]["perplexity"],
            "training_seconds": run["total_training_seconds"],
            "train_tokens": run["dataset"]["train"]["num_tokens"],
        }
    )

for row in comparison_rows:
    print(json.dumps(row, indent=2))


In [ ]:
labels = [row["tokenizer"] for row in comparison_rows]
val_ppl = [row["val_ppl"] for row in comparison_rows]
test_ppl = [row["test_ppl"] for row in comparison_rows]
vocab_sizes = [row["vocab_size"] for row in comparison_rows]
training_seconds = [row["training_seconds"] for row in comparison_rows]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].bar(labels, val_ppl, color=["#C95F44", "#2B5F75", "#D6C6A8"])
axes[0].set_title("Validation Perplexity")
axes[0].set_ylabel("Perplexity")

axes[1].bar(labels, vocab_sizes, color=["#C95F44", "#2B5F75", "#D6C6A8"])
axes[1].set_title("Vocabulary Size")
axes[1].set_ylabel("Tokens in vocabulary")

axes[2].bar(labels, training_seconds, color=["#C95F44", "#2B5F75", "#D6C6A8"])
axes[2].set_title("Training Time")
axes[2].set_ylabel("Seconds")

plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.bar(labels, test_ppl, color=["#C95F44", "#2B5F75", "#D6C6A8"])
plt.title("Test Perplexity")
plt.ylabel("Perplexity")
plt.tight_layout()
plt.show()


## Suggested report usage

From the comparison results, record at least the following for each tokenizer:
1. Vocabulary size
2. Context length / sequence length used for training
3. Training time
4. Validation perplexity
5. Test perplexity

For a fair comparison, keep the LSTM architecture the same across `word`, `char`, and `bpe` as much as possible.
